### DPA (Differential power analysis)

**this notebook cover the math behind diffrenttial power analysis explaining how to get start with it and do some examples using chipwhisprer**

### What is DPA?

**Differential Power Analysis (DPA)** is a powerful side-channel attack that exploits the dependency between the **power consumption** (or electromagnetic emission) of a cryptographic device and the data it processes.

At the transistor level, processing a larger number of '1' bits usually consumes more power (or emits more electromagnetic radiation) than processing mostly '0' bits. By carefully measuring power consumption traces while the device encrypts different known plaintexts, an attacker can use statistical methods to recover the secret key.

The core idea of DPA is to **classify** the recorded power traces based on a hypothesis about an intermediate value computed inside the algorithm, then check whether this classification reveals a statistically significant difference in power consumption. A clear difference appears **only** when the guessed key is correct.




---

Let $ A $ be a cryptographic algorithm that takes a plaintext $P$ and a secret key $K$ to produce a ciphertext $C$ Our goal is to recover the unknown secret key $K$


**Assumptions:**
- We know the structure of the algorithm and its internal operations.
- We can identify a **sensitive intermediate value** that depends on both a known plaintext and (part of) the secret key.
- We can simulate this intermediate value for every possible guess of a key byte.


####  Hypothesis Generation

For a guessed key byte $ k_b $, we compute the hypothetical intermediate value that would be produced inside the device:
$$
s(m_j, k_b).
$$

We then apply a **selection function** $ f_i $ to extract the $ i $-th bit of this value:
$$
s_i = f_i\bigl(s(m_j, k_b)\bigr) \in \{0, 1\}.
$$

####  Partitioning the Traces
We divide the power traces into two sets according to the value of the selected bit:

$$ S_{0,i} = \{p_{j}(t) | f_{i}(s(m_{j},k_{b})) = 0 \} $$
and :
$$ S_{1,i} = \{p_{j}(t) | f_{i}(s(m_{j},k_{b})) = 1 \} $$

we devise the power traces to tow sets when the selection is $1$ and when its $0$ , it's like we suppose that if we the computed values in the hardware were done using guessd key $k_{b}$ and we got $s_i = 0 $ then  the corresponding power trace $p_{j}(t)$ whihc is a small power walue and if we have the inverse like the computed value in the hardware is done using $k_{b}$ with $s_i = 1$ then the trace correspend to consmpution of $1$ (more power) .

#### Statistical Distinguisher (Difference of Means) 

so we need to define a distinguisher that confirms that such a classification is correct.

the mean of $S_{0,i}$  at $t\in [0 ,t_{end} ]$:

$$ \mu(S_{0,i}) = \frac{\sum_{j=1}^{n}f_{i}(s(m_{j},k_{b}))p_{j}(t)}{\sum_{j=1}^{n}f_{i}(s(m_{j},k_{b}))} $$

if our classification here is correct , how this quantity is going to be ? this defintion of the mean encourage the traces with $1$ value of selection to be involved in the sum and when it's $0$ to be deleted ,  so we have an important mass for this mean , we encourage to have an important value if our guess key is correct aka our classification is correct .


$$ \mu(S_{1,i}) = \frac{\sum_{j=1}^{n}(1-f_{i}(s(m_{j},k_{b})))p_{j}(t)}{\sum_{j=1}^{n}f_{i}(s(m_{j},k_{b}))} $$

here we encourage this term to be small , if  $f_{i}(s(m_{j},k_{b})) = 1$ we have  $p_{j}(t)$ this is big because we compute with $1$ hence $(1-f_{i}(s(m_{j},k_{b})))$ we cancels this value and when $f_{i}(s(m_{j},k_{b}))=0$ we have $p_{j}(t)$ small because we compute with $0$ in the correct guess and we keep it in the sum so we end up with a small $\mu(S_{1,i})$ 

we define this  :

$$D_{i}(t) = \mu(S_{0,i}) - \mu(S_{1,i})  $$

based on the previous explanation in case we compute we the correct byte key $k_{b}$ we are going to have a big value of $D_{i}(t) $ . in case we do classify randomly our traces we encourage $D_{i}(t)$ to have a small value and in all practice we have about $1000$ traces for an attack so if we don't classify correctly most of these we are going to have a flat $D_{i}$